# task_17 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

listings = pd.read_csv(base / "listings.csv")
bookings = pd.read_csv(base / "bookings.csv")
reviews = pd.read_csv(base / "reviews.csv")

bookings["checkin_date"] = pd.to_datetime(bookings["checkin_date"])
bookings["cancel_date"] = pd.to_datetime(bookings["cancel_date"])

bk = bookings.merge(listings, on="listing_id")
completed = bk[bk["canceled"] == 0].copy()

nights = completed.groupby("host_id")["nights"].sum()
listing_counts = listings.groupby("host_id")["listing_id"].nunique()
occupancy = nights / (45 * listing_counts)
q1 = str(occupancy.sort_values(ascending=False).index[0])

completed["realized_revenue"] = completed["nightly_rate"] * completed["nights"] + completed["cleaning_fee"]
q2 = str(completed.groupby("neighborhood")["realized_revenue"].sum().sort_values(ascending=False).index[0])

canceled = bk[bk["canceled"] == 1].copy()
canceled["hours_before_checkin"] = (canceled["checkin_date"] - canceled["cancel_date"]).dt.total_seconds() / 3600
q3 = f"{(100 * (canceled['hours_before_checkin'] <= 48).sum() / len(bk)):.1f}%"

revenue_per_listing_day = completed.groupby("listing_id")["realized_revenue"].sum() / 45
q4 = str(revenue_per_listing_day.sort_values(ascending=False).index[0])

review_df = reviews.merge(bookings[["booking_id", "listing_id"]], on="booking_id").merge(listings[["listing_id", "instant_book"]], on="listing_id")
diff = review_df[review_df["instant_book"] == 1]["rating"].mean() - review_df[review_df["instant_book"] == 0]["rating"].mean()
q5 = f"{diff:.3f}"


Q1: Which host_id has the highest occupancy rate, defined as total completed booked nights divided by (45 * number_of_listings_for_that_host)?

In [ ]:
print(q1)


Q2: Which neighborhood has the highest total realized revenue, where realized revenue for a completed booking is nightly_ratenights + cleaning_fee and canceled bookings contribute 0?

In [ ]:
print(q2)


Q3: What percentage of all bookings were canceled within 48 hours of checkin_date, rounded to 1 decimal place?

In [ ]:
print(q3)


Q4: Which listing_id has the highest realized revenue per available day, where available days = 45?

In [ ]:
print(q4)


Q5: What is the difference between average rating for instant-book listings and average rating for non-instant-book listings, computed as instant_book minus non_instant, rounded to 3 decimals?

In [ ]:
print(q5)
